# Healthcare Lab 2: Medical Literature Agent -- Search and Evidence Synthesis

**Series**: Agentic AI Science Playbook -- Healthcare Domain
**Goal**: Build an agent that searches medical literature and produces evidence-graded summaries.
**Prerequisites**: HC Lab 1, Foundation Labs 0-4.
**Time**: ~60 min.

---

## Background: Evidence-Based Medicine (EBM)

Evidence-based medicine integrates the best available research evidence with clinical expertise and patient values. A key challenge is navigating the volume of medical literature (PubMed indexes ~35 million articles).

### Evidence Hierarchy

```
                    ┌─────────────────────────────┐
                    │   Systematic Reviews &       │  Highest
                    │   Meta-analyses              │  quality
                    ├─────────────────────────────┤
                    │   Randomized Controlled      │
                    │   Trials (RCTs)              │
                    ├─────────────────────────────┤
                    │   Cohort Studies             │
                    │   Case-Control Studies       │
                    ├─────────────────────────────┤
                    │   Case Reports, Expert       │  Lowest
                    │   Opinion                    │  quality
                    └─────────────────────────────┘
```

---

## What You Will Build

A medical literature agent with three tools:
- `search_pubmed` -- search PubMed (simulated) with PICO query structure
- `assess_evidence_quality` -- grade evidence level for retrieved studies
- `synthesize_evidence` -- produce a clinical recommendation from multiple sources

## Learning Objectives

By the end of this lab, you will be able to:
- Build a multi-step evidence retrieval pipeline (search → assess → synthesize)
- Implement PICO-structured literature search
- Apply GRADE evidence quality assessment
- Synthesize multiple studies into actionable clinical recommendations

## Why This Matters

PubMed indexes over 35 million articles. Finding, evaluating, and synthesizing relevant evidence is the foundation of evidence-based medicine — but it takes physicians hours per clinical question.

This lab builds a **medical literature agent** that automates the complete EBM workflow:
1. **Search** using the PICO framework (the gold standard for clinical queries)
2. **Assess** each study's evidence quality using GRADE criteria
3. **Synthesize** findings into a recommendation with evidence grade

> **Impact**: Systematic reviews take 6-18 months to complete manually. AI-assisted literature agents can produce initial evidence summaries in minutes, letting researchers focus on critical appraisal rather than searching.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Completed | HC Lab 1, Foundation Labs 0-4 |
| Domain concepts | PICO, GRADE (explained below) |

**NVIDIA Connection**: This lab demonstrates the **agentic RAG** pattern — the same architecture used in NVIDIA's **AI-Q Blueprint** and **RAG Blueprint**. In production, you'd replace the simulated PubMed with real API calls and use **NeMo Retriever** for GPU-accelerated embedding and retrieval of scientific literature.

### Install Dependencies
Install the required Python packages for this lab. We need `openai` for LLM access and `pydantic` for data validation of our tool schemas.

In [ ]:
!pip install -q openai pydantic

### Connect to LLM
Set up your OpenAI or NVIDIA NIM connection. This cell detects which API you have configured and creates the client. It also imports the core libraries we will use throughout the lab.

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")
import json
from pydantic import BaseModel, Field
from typing import Literal

## Concept: The PICO Framework

PICO structures clinical questions for effective literature search:

| Component | Definition | Example |
|-----------|-----------|---------|
| **P**opulation | Who are the patients? | Adults with Type 2 diabetes |
| **I**ntervention | What treatment/exposure? | Metformin |
| **C**omparison | Compared to what? | Other first-line agents |
| **O**utcome | What is measured? | Cardiovascular events |

A well-structured PICO query dramatically improves search precision. The agent learns to decompose free-text clinical questions into PICO components.

### Define PICO-Based Search Schemas
Three tools for the evidence pipeline: `search_pubmed` uses PICO queries, `assess_evidence_quality` applies GRADE criteria, and `synthesize_evidence` combines studies into recommendations. Each tool has parameters controlling study type filters, evidence grading, and synthesis options.

## PICO Framework for Medical Queries

Structured clinical questions use PICO: Population, Intervention, Comparison, Outcome.

In [ ]:
class SearchPubMedArgs(BaseModel):
    pico_query: str = Field(..., description="Clinical question in free text or PICO format.")
    study_types: list[str] = Field(
        default_factory=lambda: ["RCT", "meta-analysis", "systematic review"],
        description="Preferred study types to retrieve.")
    max_results: int = Field(5, description="Max papers to return (1-20).")
    years_back: int = Field(5, description="Number of years back to search.")

class AssessEvidenceArgs(BaseModel):
    study_description: str = Field(..., description="Description of the study (design, sample size, outcomes).")
    study_type: str = Field(..., description="Type of study: RCT, cohort, meta-analysis, case-report, etc.")

class SynthesizeEvidenceArgs(BaseModel):
    clinical_question: str = Field(..., description="The clinical question being answered.")
    study_summaries: list[str] = Field(..., description="List of study summaries to synthesize.")
    include_grade: bool = Field(True, description="Include GRADE evidence quality rating.")

EvidenceGrade = Literal["A", "B", "C", "D"]  # A=strong, D=very low

OPENAI_TOOLS = [
    {"type": "function", "function": {
        "name": "search_pubmed",
        "description": "Search PubMed for medical literature. Use when user asks to find studies, papers, or evidence on a clinical topic.",
        "parameters": SearchPubMedArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "assess_evidence_quality",
        "description": "Assess the quality and grade of a medical study. Use when evaluating an individual study.",
        "parameters": AssessEvidenceArgs.model_json_schema()}},
    {"type": "function", "function": {
        "name": "synthesize_evidence",
        "description": "Synthesize multiple studies into a clinical recommendation with evidence grade.",
        "parameters": SynthesizeEvidenceArgs.model_json_schema()}},
]
SCHEMA_MAP = {
    "search_pubmed": SearchPubMedArgs,
    "assess_evidence_quality": AssessEvidenceArgs,
    "synthesize_evidence": SynthesizeEvidenceArgs,
}
print("Medical literature tools:", [t["function"]["name"] for t in OPENAI_TOOLS])

## Simulated PubMed Database

In production, connect to the real PubMed API using `requests` or the `pymed` library.

### Load Simulated PubMed Database
In production, you would connect to PubMed's real API. For this lab, we use a curated set of realistic papers with study types, sample sizes, and abstracts. The simulated database covers metformin/diabetes and aspirin/heart disease topics.

In [ ]:
SIMULATED_PAPERS = {
    "metformin diabetes": [
        {"pmid": "34521234", "title": "Metformin and cardiovascular outcomes in type 2 diabetes: a meta-analysis of 20 RCTs", "authors": "Smith J et al.", "year": 2023, "journal": "NEJM", "study_type": "meta-analysis", "n": 45000, "abstract": "Meta-analysis of 20 RCTs showed metformin reduces MACE by 14% (RR 0.86, 95% CI 0.79-0.93) vs. placebo in T2DM patients. NNT=47 over 5 years."},
        {"pmid": "33891234", "title": "Metformin vs. SGLT2i as first-line therapy: a head-to-head RCT", "authors": "Jones A et al.", "year": 2022, "journal": "Lancet", "study_type": "RCT", "n": 3200, "abstract": "RCT comparing metformin to empagliflozin in drug-naive T2DM. HbA1c reduction similar (-1.1% vs -1.3%), but SGLT2i superior for weight loss (-3.1 vs -1.2 kg, p<0.001)."},
        {"pmid": "32109876", "title": "Long-term safety of metformin: 10-year cohort study", "authors": "Brown K et al.", "year": 2021, "journal": "BMJ", "study_type": "cohort", "n": 12000, "abstract": "10-year follow-up showed no increased risk of lactic acidosis with eGFR >30. B12 deficiency in 6.3% of long-term users; monitoring recommended."},
    ],
    "aspirin heart disease": [
        {"pmid": "35123456", "title": "Aspirin for primary cardiovascular prevention: updated meta-analysis", "authors": "Garcia R et al.", "year": 2023, "journal": "JAMA", "study_type": "meta-analysis", "n": 165000, "abstract": "Primary prevention: aspirin reduces MI (RR 0.89) but increases major bleeding (RR 1.47). Net benefit unclear in low-risk individuals. Guidelines now recommend against routine use in primary prevention."},
        {"pmid": "34567890", "title": "Low-dose aspirin in secondary prevention: systematic review", "authors": "Wang L et al.", "year": 2022, "journal": "Circulation", "study_type": "systematic review", "n": 30000, "abstract": "Strong evidence supports aspirin 75-100mg daily for secondary prevention after MI or stroke. NNT=50 over 3 years for preventing recurrent events."},
    ],
}

def simulated_pubmed_search(query: str, study_types: list, max_results: int, years_back: int) -> list:
    key = next((k for k in SIMULATED_PAPERS if any(w in query.lower() for w in k.split())), None)
    results = SIMULATED_PAPERS.get(key, [])
    if study_types:
        filtered = [p for p in results if any(st.lower() in p["study_type"].lower() for st in study_types)]
        results = filtered or results
    return results[:max_results]

print("Simulated PubMed database loaded.")
print("Sample queries:", list(SIMULATED_PAPERS.keys()))

## Tool Implementations

### Implement Literature Tools
The search tool queries our simulated database. The evidence assessor grades study quality (A-D) using a combination of rule-based mapping and LLM analysis. The synthesizer combines multiple studies into a clinical recommendation with evidence grade and caveats.

In [ ]:
def execute_search_pubmed(args: SearchPubMedArgs) -> str:
    papers = simulated_pubmed_search(args.pico_query, args.study_types, args.max_results, args.years_back)
    if not papers:
        return json.dumps({"results": [], "note": "No papers found. Try broadening the query."})
    return json.dumps({
        "query": args.pico_query,
        "n_results": len(papers),
        "results": [{
            "pmid": p["pmid"],
            "title": p["title"],
            "authors": p["authors"],
            "year": p["year"],
            "journal": p["journal"],
            "study_type": p["study_type"],
            "n_patients": p["n"],
            "abstract_excerpt": p["abstract"][:200],
        } for p in papers]
    }, indent=2)

EVIDENCE_GRADES = {
    "meta-analysis": "A", "systematic review": "A",
    "RCT": "B", "randomized": "B",
    "cohort": "C", "case-control": "C",
    "case-report": "D", "expert opinion": "D", "case series": "D",
}

def execute_assess_evidence(args: AssessEvidenceArgs) -> str:
    grade = next((g for k, g in EVIDENCE_GRADES.items() if k in args.study_type.lower()), "C")
    prompt = (
        f"Assess this medical study:\nType: {args.study_type}\nDescription: {args.study_description}\n"
        f"Assign evidence grade (A/B/C/D) and list 2 strengths and 2 limitations.\n"
        f"Return JSON: {{\"grade\": ..., \"strengths\": [...], \"limitations\": [...], \"clinical_relevance\": ...}}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=400,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    result = json.loads(r.choices[0].message.content or "{}")
    result["suggested_grade"] = grade
    return json.dumps(result, indent=2)

def execute_synthesize_evidence(args: SynthesizeEvidenceArgs) -> str:
    prompt = (
        f"Clinical question: {args.clinical_question}\n"
        f"Available evidence:\n" + "\n".join(f"  {i+1}. {s}" for i, s in enumerate(args.study_summaries)) +
        f"\n\nSynthesize into a clinical recommendation. "
        f"{'Include GRADE evidence quality (A-D). ' if args.include_grade else ''}"
        f"Return JSON: {{\"recommendation\": ..., \"evidence_grade\": ..., \"key_findings\": [...], \"caveats\": [...]}}"
    )
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0, max_tokens=600,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content or "{}"

def run_tool(name, args):
    if name == "search_pubmed": return execute_search_pubmed(args)
    if name == "assess_evidence_quality": return execute_assess_evidence(args)
    if name == "synthesize_evidence": return execute_synthesize_evidence(args)
    return f"[error] Unknown: {name}"

MED_LIT_SYSTEM = (
    "You are a medical literature assistant trained in evidence-based medicine. "
    "Search for studies, assess evidence quality, and synthesize clinical recommendations. "
    "Always note evidence limitations. Use the provided tools."
)

def med_lit_agent(user_message):
    r = client.chat.completions.create(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": MED_LIT_SYSTEM},
                   {"role": "user", "content": user_message}],
        tools=OPENAI_TOOLS, tool_choice="auto")
    msg = r.choices[0].message
    if not msg.tool_calls:
        return {"tool": None, "content": msg.content}
    call = msg.tool_calls[0]
    validated = SCHEMA_MAP[call.function.name](**json.loads(call.function.arguments))
    return {"tool": call.function.name, "args": validated}

## Experiment 1: Multi-step Evidence Retrieval Pipeline

### Experiment 1, Step 1: Search Literature
Ask a PICO-structured clinical question about metformin for Type 2 diabetes. The agent searches for relevant studies using our simulated PubMed database. This is the first step of the three-step evidence retrieval pipeline.

In [ ]:
clinical_question = (
    "In adults with type 2 diabetes, does metformin compared to other first-line agents "
    "reduce cardiovascular events with acceptable safety?"
)

print("=== Step 1: Search Literature ===")
r = med_lit_agent(f"Search for: {clinical_question}")
if r["tool"]:
    search_results = json.loads(run_tool(r["tool"], r["args"]))
    print(f"Found {search_results['n_results']} papers:")
    for paper in search_results["results"]:
        print(f"  [{paper['year']}] {paper['title'][:70]} ({paper['study_type']}, N={paper['n_patients']})")

<details>
<summary>Expected output (click to expand)</summary>

```
=== Step 1: Search Literature ===
Found 3 papers:
  [2023] Metformin and cardiovascular outcomes in type 2 diabetes: a meta-an (meta-analysis, N=45000)
  [2022] Metformin vs. SGLT2i as first-line therapy: a head-to-head RCT (RCT, N=3200)
  [2021] Long-term safety of metformin: 10-year cohort study (cohort, N=12000)
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment 1, Step 2: Assess Evidence Quality
For each paper found, the agent evaluates the evidence grade based on study design, sample size, and findings. Meta-analyses and systematic reviews receive higher grades (A) while cohort studies receive lower grades (C). The LLM also identifies strengths and limitations.

In [ ]:
print("\n=== Step 2: Assess Evidence Quality ===")
study_summaries = []
for paper in search_results.get("results", []):
    r2 = med_lit_agent(
        f"Assess evidence quality for this study: {paper['title']}. "
        f"Study type: {paper['study_type']}. N={paper['n_patients']}. "
        f"Abstract: {paper['abstract_excerpt']}")
    if r2["tool"]:
        assessment = json.loads(run_tool(r2["tool"], r2["args"]))
        print(f"  {paper['title'][:55]}...")
        print(f"    Grade={assessment.get('grade','?')} | {assessment.get('clinical_relevance','')[:80]}")
        study_summaries.append(f"{paper['title']} (Grade {assessment.get('grade','?')}): {paper['abstract_excerpt'][:100]}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== Step 2: Assess Evidence Quality ===
  Metformin and cardiovascular outcomes in type 2 diabetes...
    Grade=A | Strong meta-analysis with large sample size...
  Metformin vs. SGLT2i as first-line therapy: a head-to-h...
    Grade=B | Well-designed RCT with adequate power...
  Long-term safety of metformin: 10-year cohort study...
    Grade=C | Long follow-up but observational design limits causal inference...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### Experiment 1, Step 3: Synthesize Evidence
Combine all assessed studies into a single clinical recommendation with an overall evidence grade. The synthesizer weighs the individual study grades, identifies key findings, and notes caveats. This completes the three-step evidence retrieval pipeline.

In [ ]:
print("\n=== Step 3: Synthesize Evidence ===")
r3 = med_lit_agent(
    f"Synthesize evidence to answer: {clinical_question}. "
    f"Studies: {study_summaries}")
if r3["tool"]:
    synthesis = json.loads(run_tool(r3["tool"], r3["args"]))
    print("CLINICAL RECOMMENDATION:")
    print(f"  Grade: {synthesis.get('evidence_grade','?')}")
    print(f"  Recommendation: {synthesis.get('recommendation','?')}")
    print("  Key findings:")
    for f in synthesis.get("key_findings", []):
        print(f"    - {f}")
    print("  Caveats:")
    for c in synthesis.get("caveats", []):
        print(f"    - {c}")

<details>
<summary>Expected output (click to expand)</summary>

```
=== Step 3: Synthesize Evidence ===
CLINICAL RECOMMENDATION:
  Grade: A
  Recommendation: Metformin remains a strong first-line therapy for type 2
    diabetes with cardiovascular risk. Meta-analysis shows 14% reduction in
    MACE. SGLT2 inhibitors show comparable glycemic control with superior
    weight loss benefits.
  Key findings:
    - Metformin reduces MACE by 14% (RR 0.86, 95% CI 0.79-0.93)
    - SGLT2i superior for weight loss (-3.1 vs -1.2 kg)
    - No increased lactic acidosis risk with eGFR >30
  Caveats:
    - B12 deficiency in 6.3% of long-term users
    - Head-to-head data limited to single RCT
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## Reflection Questions

1. **The evidence synthesis combined 3 studies.** What are the risks of synthesizing heterogeneous studies? How does GRADE help mitigate this?
2. **The simulated PubMed returned perfect results.** In reality, most searches return irrelevant papers. How would you add a relevance filtering step?
3. **This lab used a 3-step pipeline (search → assess → synthesize).** How would you model this as a LangGraph state machine (from Lab 4) with error recovery?

## Summary

| Tool | Capability |
|------|-----------|
| `search_pubmed` | PICO-structured literature search with study type filtering |
| `assess_evidence_quality` | GRADE-based evidence quality assessment |
| `synthesize_evidence` | Multi-study synthesis into clinical recommendations |

**Next**: HC Lab 3 -- Clinical Trial Assistant for protocol analysis and eligibility checking.

---
*Agentic AI Science Playbook -- Healthcare Domain, Lab HC2.*